# 🔷 TypeScript 30天核心训练## 每天：📖示例 → ✏️练习 → 🧪类型测试### by CloudClaw 🦞

In [ ]:
// 🧪 测试辅助 (Node.js + ts-node 或 Deno 环境)// TypeScript 需要编译运行: npx ts-node this_file.ts// 或用在线 Playground: https://www.typescriptlang.org/playfunction assert(condition: boolean, msg?: string): asserts condition {  if (!condition) throw new Error(`❌ ${msg || "assertion failed"}`);}type TestFn = () => void;const tests: { name: string; fn: TestFn }[] = [];function test(name: string, fn: TestFn) {  tests.push({ name, fn });}function runTests() {  let passed = 0, failed = 0;  for (const t of tests) {    try { t.fn(); passed++; console.log(`  ✅ ${t.name}`); }    catch(e) { failed++; console.log(`  ❌ ${t.name}: ${e}`); }  }  console.log(`\n📊 ${passed}/${passed+failed} passed`);}console.log("✅ TypeScript 测试工具就绪");

# Day 1 — 基础类型注释---

## 📖 TypeScript 类型系统入门```ts// 基本类型注释let name: string = "小明";let age: number = 25;let isStudent: boolean = true;let scores: number[] = [85, 90, 78];let tuple: [string, number] = ["小明", 25];// 类型推断 — TS 自动推断类型let city = "北京";   // TS 推断为 string，无需写 :stringlet count = 42;       // 推断为 number// 联合类型let id: string | number = "abc123";id = 456;  // ✅ OK// id = true; // ❌ 类型错误// any — 关闭类型检查 (避免使用)let data: any = "anything";data = 42;  // 无检查// unknown — 安全的 anylet input: unknown = "hello";// input.toUpperCase(); // ❌ 需先类型收窄if (typeof input === "string") {  console.log(input.toUpperCase()); // ✅ 安全}```**核心原则**: 能用类型推断就不写类型注释，多用 const。

## ✏️ 练习### 练习1声明变量: name(string), age(number), hobbies(string[]), id(string | number)### 练习2写 `describe(value: unknown): string` 函数，根据 typeof 返回不同描述### 练习3写 `firstElement(arr: T[]): T | undefined` 泛型函数 (Day3才学，先写具体版本: number[] 版)

In [ ]:
// 练习1const username: string = "测试员";const userAge: number = 25;const hobbies: string[] = ["编程", "读书"];const userId: string | number = "u_123";assert(typeof username === "string");assert(Array.isArray(hobbies));// 练习2function describe(value: unknown): string {  if (typeof value === "string") return `字符串: "${value}"`;  if (typeof value === "number") return `数字: ${value}`;  if (typeof value === "boolean") return `布尔值: ${value}`;  if (Array.isArray(value)) return `数组，长度: ${value.length}`;  if (value === null) return "null";  if (value === undefined) return "undefined";  return `其他: ${typeof value}`;}test("describe string", () => assert(describe("hello").includes("字符串")));test("describe number", () => assert(describe(42) === "数字: 42"));test("describe array", () => assert(describe([1,2,3]).includes("3")));test("describe null", () => assert(describe(null) === "null"));// 练习3function firstElement(arr: number[]): number | undefined {  return arr.length > 0 ? arr[0] : undefined;}test("firstElement with array", () => assert(firstElement([1,2,3]) === 1));test("firstElement empty array", () => assert(firstElement([]) === undefined));runTests();

# Day 2 — 接口 (Interface)---

## 📖 Interface vs Type```ts// 接口 — 定义对象形状interface User {  name: string;  age: number;  email?: string;       // ? 可选属性  readonly id: number;  // readonly 只读}const user: User = {  name: "小明",  age: 25,  id: 1  // email 可省略};// user.id = 2; // ❌ 只读// 接口扩展interface Admin extends User {  role: "admin" | "superadmin";  permissions: string[];}// Type 别名 — 更灵活type Status = "active" | "inactive" | "banned";type Point = { x: number; y: number };type ID = string | number;// Interface vs Type 区别// - Interface 可被合并 (declaration merging)// - Type 可用于联合类型/元组// - 优先用 interface，需要联合类型时用 type```

## ✏️ 练习### 练习1定义 Product 接口: id(number), name(string), price(number), category?(string)### 练习2用 extends 定义 Electronics 接口扩展 Product: warranty(number), brand(string)### 练习3定义 type 别名: Result<T> = { success: true; data: T } | { success: false; error: string }

In [ ]:
// 练习1interface Product {  id: number;  name: string;  price: number;  category?: string;}const p1: Product = { id: 1, name: "Laptop", price: 5999 };test("Product interface", () => {  assert(p1.id === 1 && p1.name === "Laptop");});// 练习2interface Electronics extends Product {  warranty: number;  brand: string;}const phone: Electronics = {  id: 2, name: "Phone", price: 4999,  warranty: 12, brand: "BrandX"};test("Electronics extends Product", () => {  assert(phone.warranty === 12 && phone.brand === "BrandX");});// 练习3type Result<T> =   | { success: true; data: T }  | { success: false; error: string };function handleResult(result: Result<number>): string {  if (result.success) return `数据: ${result.data}`;  return `错误: ${result.error}`;}test("Result type success", () => {  assert(handleResult({ success: true, data: 42 }) === "数据: 42");});test("Result type error", () => {  assert(handleResult({ success: false, error: "失败" }).includes("错误"));});runTests();

# Day 3 — 泛型 (Generics)---

## 📖 泛型```ts// 泛型函数function identity<T>(arg: T): T {  return arg;}identity<string>("hello");  // 显式指定identity(42);               // TS 自动推断// 泛型约束 extendsfunction getLength<T extends { length: number }>(arg: T): number {  return arg.length;}getLength("hello");    // 5getLength([1,2,3]);    // 3// getLength(123);     // ❌ number 没有 length// 泛型接口interface KeyValue<K, V> {  key: K;  value: V;}// 泛型类class Stack<T> {  private items: T[] = [];  push(item: T) { this.items.push(item); }  pop(): T | undefined { return this.items.pop(); }  peek(): T | undefined { return this.items[this.items.length - 1]; }}const numStack = new Stack<number>();numStack.push(1); numStack.push(2);```

## ✏️ 练习### 练习1泛型函数 `first<T>(arr: T[]): T | undefined` 返回第一个元素### 练习2泛型函数 `pair<T, U>(a: T, b: U): [T, U]` 返回元组### 练习3泛型类 `Queue<T>`: enqueue, dequeue, peek, size

In [ ]:
// 练习1function best<T>(arr: T[]): T | undefined {  return arr[0];}test("first with numbers", () => assert(best([1,2,3]) === 1));test("first with strings", () => assert(best(["a","b"]) === "a"));test("first empty", () => assert(best([]) === undefined));// 练习2function pair<T, U>(a: T, b: U): [T, U] {  return [a, b];}test("pair", () => {  const [s, n] = pair("age", 25);  assert(s === "age" && n === 25);});// 练习3class Queue<T> {  private items: T[] = [];  enqueue(item: T) { this.items.push(item); }  dequeue(): T | undefined { return this.items.shift(); }  peek(): T | undefined { return this.items[0]; }  get size(): number { return this.items.length; }}test("Queue", () => {  const q = new Queue<number>();  q.enqueue(1); q.enqueue(2); q.enqueue(3);  assert(q.dequeue() === 1);  assert(q.peek() === 2);  assert(q.size === 2);});runTests();

# Day 4 — 联合类型与交叉类型---

## 📖 Union & Intersection Types```ts// 联合类型 (|) — 值可以是多种类型之一type StringOrNumber = string | number;function printId(id: StringOrNumber) {  if (typeof id === "string") console.log(id.toUpperCase());  else console.log(id.toFixed(2));}// 字面量联合类型type Direction = "up" | "down" | "left" | "right";type HTTPMethod = "GET" | "POST" | "PUT" | "DELETE";// 交叉类型 (&) — 合并多个类型type Draggable = { drag: () => void };type Resizable = { resize: () => void };type UIWidget = Draggable & Resizable;const widget: UIWidget = {  drag: () => console.log("dragging"),  resize: () => console.log("resizing")};// 实际场景: 合并 propstype WithId = { id: number };type WithTimestamp = { createdAt: Date };type Entity = WithId & WithTimestamp;```

## ✏️ 练习### 练习1定义 `parseValue(value: string | number | boolean): string` 转为字符串表示### 练习2定义 Color 类型: "red" | "green" | "blue" | `#${string}` (模板字面量类型)### 练习3Person = HasName & HasAge & HasContact，三个接口交叉合并

In [ ]:
// 练习1function parseValue(value: string | number | boolean): string {  if (typeof value === "boolean") return value ? "是" : "否";  return String(value);}test("parseValue string", () => assert(parseValue("hello") === "hello"));test("parseValue number", () => assert(parseValue(42) === "42"));test("parseValue boolean", () => assert(parseValue(true) === "是"));// 练习2 — 模板字面量类型type Color = "red" | "green" | "blue" | `#${string}`;function setColor(c: Color): string { return `color: ${c}`; }test("Color literal", () => assert(setColor("red") === "color: red"));test("Color hex", () => assert(setColor("#ff0000") === "color: #ff0000"));// 练习3interface HasName { name: string; }interface HasAge { age: number; }interface HasContact { email: string; phone?: string; }type Person = HasName & HasAge & HasContact;const person: Person = { name: "小明", age: 25, email: "ming@test.com" };test("intersection Person", () => {  assert(person.name === "小明" && person.age === 25 && person.email.includes("@"));});runTests();

# Day 5 — 枚举与常量---

## 📖 Enum & const```ts// 数字枚举enum Direction {  Up,      // 0  Down,    // 1  Left,    // 2  Right    // 3}console.log(Direction.Up);       // 0console.log(Direction[0]);       // "Up"// 字符串枚举 (推荐)enum Status {  Active = "ACTIVE",  Inactive = "INACTIVE",  Banned = "BANNED"}// const enum — 编译时内联 (零运行时开销)const enum Size {  Small = 1,  Medium = 2,  Large = 3}// as const — 不可变断言const config = {  apiUrl: "https://api.example.com",  timeout: 5000} as const;// config.apiUrl = "other"; // ❌ readonly// 从数组推导联合类型const FRUITS = ["apple", "banana", "orange"] as const;type Fruit = typeof FRUITS[number]; // "apple" | "banana" | "orange"```

## ✏️ 练习### 练习1定义 OrderStatus 字符串枚举: Pending / Processing / Shipped / Delivered### 练习2`as const` 定义 HTTP 状态码常量对象，从值推导类型### 练习3写 `getStatusLabel(status: OrderStatus): string` 返回中文标签

In [ ]:
// 练习1enum OrderStatus {  Pending = "PENDING",  Processing = "PROCESSING",  Shipped = "SHIPPED",  Delivered = "DELIVERED"}// 练习2const HTTP_STATUS = {  OK: 200,  NotFound: 404,  ServerError: 500} as const;type HTTPStatusCode = typeof HTTP_STATUS[keyof typeof HTTP_STATUS]; // 200 | 404 | 500// 练习3function getStatusLabel(status: OrderStatus): string {  switch (status) {    case OrderStatus.Pending: return "待处理";    case OrderStatus.Processing: return "处理中";    case OrderStatus.Shipped: return "已发货";    case OrderStatus.Delivered: return "已送达";    default: return "未知";  }}test("enum labels", () => {  assert(getStatusLabel(OrderStatus.Pending) === "待处理");  assert(getStatusLabel(OrderStatus.Delivered) === "已送达");});test("as const types", () => {  assert(HTTP_STATUS.OK === 200 && HTTP_STATUS.NotFound === 404);});runTests();

# Day 6 — 函数类型---

## 📖 函数签名与重载```ts// 函数类型表达式type MathOp = (a: number, b: number) => number;const add: MathOp = (a, b) => a + b;// 调用签名 (对象可调用)interface Calculator {  (a: number, b: number): number;  description: string;}// 可选参数和默认值function greet(name: string, greeting: string = "你好"): string {  return `${greeting}, ${name}`;}// 剩余参数function sum(...nums: number[]): number {  return nums.reduce((a,b) => a + b, 0);}// 函数重载function process(value: string): string;function process(value: number): number;function process(value: string | number): string | number {  if (typeof value === "string") return value.toUpperCase();  return value * 2;}```

## ✏️ 练习### 练习1定义 `Calculator` 函数类型: (a:number, b:number, op:string) => number### 练习2写函数重载 `format(input: string): string` 和 `format(input: number): string`### 练习3实现 `pipe<T>(...fns: Array<(arg: T) => T>): (arg: T) => T`

In [ ]:
// 练习1type Calculator = (a: number, b: number, op: string) => number;const calc: Calculator = (a, b, op) => {  switch(op) {    case "+": return a + b;    case "-": return a - b;    case "*": return a * b;    case "/": return b !== 0 ? a / b : NaN;    default: return NaN;  }};test("calculator", () => {  assert(calc(10, 5, "+") === 15);  assert(calc(10, 5, "*") === 50);});// 练习2: 重载 — 用联合类型实现function format(input: string): string;function format(input: number): string;function format(input: string | number): string {  if (typeof input === "string") return input.trim();  return input.toFixed(2);}test("format string", () => assert(format("  hello  ") === "hello"));test("format number", () => assert(format(3.14159) === "3.14"));// 练习3: pipefunction pipe<T>(...fns: Array<(arg: T) => T>): (arg: T) => T {  return (arg: T) => fns.reduce((value, fn) => fn(value), arg);}const add1 = (x: number) => x + 1;const double = (x: number) => x * 2;const piped = pipe(add1, double);test("pipe", () => assert(piped(3) === 8)); // (3+1)*2runTests();

# Day 7 — 类与访问修饰符---

## 📖 TypeScript 类```tsclass Animal {  public name: string;           // public (默认)  private _age: number;          // 仅类内部  protected species: string;     // 类+子类  readonly id: number;           // 只读  constructor(name: string, age: number) {    this.name = name;    this._age = age;    this.species = "未知";    this.id = Math.random();  }  // getter/setter  get age(): number { return this._age; }  set age(value: number) {    if (value < 0) throw new Error("年龄无效");    this._age = value;  }  // 静态方法  static create(name: string): Animal {    return new Animal(name, 0);  }}// 抽象类abstract class Shape {  abstract area(): number;  describe(): string { return `面积: ${this.area()}`; }}```

## ✏️ 练习### 练习1BankAccount 类: private balance, deposit/withdraw, getter balance### 练习2抽象类 Shape → Rectangle/Circle 子类实现 area()### 练习3Logger 抽象类 → ConsoleLogger/FileLogger 子类

In [ ]:
// 练习1class BankAccount {  private _balance: number;  readonly owner: string;  constructor(owner: string, initial: number = 0) {    this.owner = owner;    this._balance = initial;  }  deposit(amount: number): number {    this._balance += amount;    return this._balance;  }  withdraw(amount: number): number {    if (amount > this._balance) throw new Error("余额不足");    this._balance -= amount;    return this._balance;  }  get balance(): number { return this._balance; }}test("BankAccount", () => {  const acc = new BankAccount("小明", 1000);  acc.deposit(500);  assert(acc.balance === 1500);  acc.withdraw(300);  assert(acc.balance === 1200);  assert(acc.owner === "小明");});// 练习2abstract class Shape {  abstract area(): number;  describe(): string { return `面积: ${this.area()}`; }}class Rectangle extends Shape {  constructor(private w: number, private h: number) { super(); }  area(): number { return this.w * this.h; }}class Circle extends Shape {  constructor(private r: number) { super(); }  area(): number { return Math.PI * this.r * this.r; }}test("Shape polymorphism", () => {  const shapes: Shape[] = [new Rectangle(4, 5), new Circle(3)];  assert(shapes[0].area() === 20);  assert(Math.abs(shapes[1].area() - 28.27) < 0.1);  assert(shapes[0].describe().includes("面积"));});// 练习3abstract class Logger {  abstract log(msg: string): void;  warn(msg: string) { this.log(`[WARN] ${msg}`); }}class ConsoleLogger extends Logger {  private logs: string[] = [];  log(msg: string) { this.logs.push(msg); }  getLogs(): string[] { return [...this.logs]; }}test("Abstract Logger", () => {  const l = new ConsoleLogger();  l.log("test"); l.warn("alert");  const logs = l.getLogs();  assert(logs.length === 2 && logs[1].includes("WARN"));});runTests();

# Day 8 — Utility Types---

## 📖 TypeScript 内置工具类型```tsinterface User {  id: number;  name: string;  email: string;  age: number;}// Partial<T> — 所有属性可选type PartialUser = Partial<User>;// { id?: number; name?: string; ... }// Required<T> — 所有属性必选type RequiredUser = Required<Partial<User>>;// Pick<T, K> — 选取指定属性type UserName = Pick<User, "name" | "email">;// Omit<T, K> — 排除指定属性type UserWithoutId = Omit<User, "id">;// Readonly<T> — 所有属性只读type ReadonlyUser = Readonly<User>;// Record<K, V> — 键值对映射type PageInfo = Record<string, string>;// ReturnType<T> — 获取函数返回值类型function getData() { return { id: 1, name: "test" }; }type Data = ReturnType<typeof getData>;```

## ✏️ 练习### 练习1用 Partial 创建一个 `updateUser(id, partial: Partial<User>)` 函数### 练习2用 Pick 和 Omit: 从 Product 类型创建 ProductSummary(只有name,price)、ProductForm(除id外)### 练习3用 Record: type Translation = Record<"en"|"zh"|"ja", string>

In [ ]:
interface User2 { id: number; name: string; email: string; }// 练习1function updateUser(id: number, partial: Partial<User2>): User2 {  const user: User2 = { id, name: "默认", email: "" };  return { ...user, ...partial };}test("Partial update", () => {  const updated = updateUser(1, { name: "小明" });  assert(updated.id === 1 && updated.name === "小明");});// 练习2interface Product2 { id: number; name: string; price: number; category: string; }type ProductSummary = Pick<Product2, "name" | "price">;type ProductForm = Omit<Product2, "id">;const summary: ProductSummary = { name: "Laptop", price: 5999 };const form: ProductForm = { name: "Phone", price: 4999, category: "电子" };test("Pick and Omit", () => {  assert(summary.name === "Laptop" && form.category === "电子");});// 练习3type Translation = Record<"en" | "zh" | "ja", string>;const greeting: Translation = { en: "Hello", zh: "你好", ja: "こんにちは" };test("Record translation", () => {  assert(greeting.zh === "你好" && greeting.ja === "こんにちは");});runTests();

# Day 9 — 类型守卫与收窄---

## 📖 Type Guards & Narrowing```ts// typeof 收窄function process(value: string | number) {  if (typeof value === "string") {    // value 此处是 string    return value.toUpperCase();  }  // value 此处是 number  return value * 2;}// instanceof 收窄class Dog { bark() { return "汪汪"; } }class Cat { meow() { return "喵"; } }function speak(animal: Dog | Cat) {  if (animal instanceof Dog) return animal.bark();  return animal.meow();}// 自定义类型守卫function isString(value: unknown): value is string {  return typeof value === "string";}// in 操作符收窄interface Fish { swim: () => void }interface Bird { fly: () => void }function move(animal: Fish | Bird) {  if ("swim" in animal) animal.swim();  else animal.fly();}```

## ✏️ 练习### 练习1`processValue(value: string | number | boolean): string` 用 typeof 收窄### 练习2自定义类型守卫 `isError(value: unknown): value is Error`### 练习3`formatResult(result: TSuccess | TError)` 用 discriminated union 区分

In [ ]:
// 练习1function processValue(value: string | number | boolean): string {  if (typeof value === "string") return value.trim();  if (typeof value === "number") return value.toFixed(2);  return value ? "是" : "否";}test("processValue string", () => assert(processValue(" hi ") === "hi"));test("processValue number", () => assert(processValue(3.14159) === "3.14"));// 练习2function isError(value: unknown): value is Error {  return value instanceof Error;}test("isError true", () => assert(isError(new Error("test")) === true));test("isError false", () => assert(isError("not error") === false));// 练习3: discriminated unioninterface Success { kind: "success"; data: string }interface Failure { kind: "failure"; message: string }type ApiResult = Success | Failure;function formatResult(result: ApiResult): string {  switch (result.kind) {    case "success": return `✅ ${result.data}`;    case "failure": return `❌ ${result.message}`;  }}test("discriminated union success", () => {  const r = formatResult({ kind: "success", data: "done" });  assert(r === "✅ done");});test("discriminated union failure", () => {  const r = formatResult({ kind: "failure", message: "error" });  assert(r === "❌ error");});runTests();

# Day 10 — Promise 与 Async---

## 📖 Promise<T> 类型```ts// Promise<T> 泛型指定 resolve 类型function fetchUser(id: number): Promise<{ id: number; name: string }> {  return Promise.resolve({ id, name: `User${id}` });}// async 函数返回 Promise<T>async function getUser(id: number): Promise<string> {  const user = await fetchUser(id);  return user.name;}// 错误处理async function safeFetch(url: string): Promise<any> {  try {    const response = await fetch(url);    if (!response.ok) throw new Error(`HTTP ${response.status}`);    return response.json();  } catch (error) {    // error 默认是 unknown，需要类型收窄    if (error instanceof Error) {      console.error(error.message);    }    return null;  }}// Promise.all 类型推导async function loadAll(): Promise<[string, number]> {  const [a, b] = await Promise.all([    Promise.resolve("hello"),    Promise.resolve(42)  ]);  return [a, b];}```

## ✏️ 练习### 练习1`delay(ms): Promise<string>` 返回 delay 毫秒后 resolve 的消息### 练习2`retry<T>(fn: () => Promise<T>, times: number): Promise<T>` 失败重试### 练习3`fetchWithTimeout<T>(promise: Promise<T>, timeoutMs: number): Promise<T>` 超时控制

In [ ]:
// 练习1function delay(ms: number): Promise<string> {  return new Promise(resolve => setTimeout(() => resolve(`完成 ${ms}ms`), ms));}// 用立即完成验证function instantDelay(): Promise<string> { return Promise.resolve("instant"); }// 练习2: retryasync function retry<T>(fn: () => Promise<T>, times: number): Promise<T> {  let lastError: unknown;  for (let i = 0; i < times; i++) {    try {      return await fn();    } catch (e) {      lastError = e;      if (i === times - 1) throw lastError;    }  }  throw lastError;}// 练习3: 超时function fetchWithTimeout<T>(promise: Promise<T>, ms: number): Promise<T> {  const timeout = new Promise<never>((_, reject) =>    setTimeout(() => reject(new Error(`超时 ${ms}ms`)), ms)  );  return Promise.race([promise, timeout]);}// 快速同步测试(async () => {  test("delay returns string", async () => {    const result = await instantDelay();    assert(result === "instant");  });  test("retry succeeds", async () => {    let callCount = 0;    const fn = async () => { callCount++; if (callCount < 2) throw new Error("fail"); return "ok"; };    const result = await retry(fn, 3);    assert(result === "ok" && callCount === 2);  });  test("fetchWithTimeout resolves", async () => {    const result = await fetchWithTimeout(Promise.resolve("done"), 1000);    assert(result === "done");  });  runTests();})();

# Day 11 — 泛型约束与条件类型---

## 📖 高级泛型```ts// extends 约束function getProperty<T, K extends keyof T>(obj: T, key: K): T[K] {  return obj[key];}const user = { name: "小明", age: 25 };getProperty(user, "name");  // "小明" (类型是 string)// getProperty(user, "email"); // ❌ 编译错误// 条件类型type IsString<T> = T extends string ? "yes" : "no";type A = IsString<"hello">;  // "yes"type B = IsString<42>;       // "no"// infer — 在条件类型中推断type Unwrap<T> = T extends Promise<infer U> ? U : T;type C = Unwrap<Promise<string>>;  // stringtype D = Unwrap<number>;           // number// 映射类型type Nullable<T> = { [K in keyof T]: T[K] | null };type NullableUser = Nullable<{ name: string; age: number }>;// { name: string | null; age: number | null }```

## ✏️ 练习### 练习1`getProperty` 泛型函数: 用 `keyof` 约束 key 必须是 obj 的属性### 练习2条件类型 `IsArray<T>`: T extends any[] → "array" : "not array"### 练习3映射类型 `Readonly<T>`: { readonly [K in keyof T]: T[K] }

In [ ]:
// 练习1function getProperty<T, K extends keyof T>(obj: T, key: K): T[K] {  return obj[key];}test("getProperty typed", () => {  const user = { name: "小明", age: 25 };  assert(getProperty(user, "name") === "小明");  assert(getProperty(user, "age") === 25);});// 练习2: 条件类型type IsArray<T> = T extends any[] ? "array" : "not array";// 类型级测试 (编译时)const testIsArray: true = true;test("IsArray type concept", () => assert(testIsArray));// 练习3: 手写 Readonly 映射类型// type MyReadonly<T> = { readonly [K in keyof T]: T[K] };interface TestObj { a: string; b: number; }// type ReadonlyTest = MyReadonly<TestObj>;// 运行时验证function freeze<T extends object>(obj: T): Readonly<T> {  return obj; // 实际应 Object.freeze}test("Readonly concept", () => {  const obj = freeze({ a: 1, b: "hello" });  assert(obj.a === 1 && obj.b === "hello");});runTests();

# Day 12 — 声明文件 (.d.ts)---

## 📖 Declaration Files```ts// === globals.d.ts ===// 为没有类型的 JS 库提供类型声明declare module "my-library" {  export function doSomething(x: number): string;  export const VERSION: string;}// 扩展全局类型declare global {  interface Window {    myApp: { version: string };  }  const APP_CONFIG: {    apiUrl: string;    debug: boolean;  };}// 模块声明通配符declare module "*.svg" {  const content: string;  export default content;}declare module "*.css" {  const content: Record<string, string>;  export default content;}// 实用: declare 第三方 window 属性// 放到项目的 global.d.ts 中```

## ✏️ 练习### 练习1为 "math-utils" 模块写声明: export add, subtract, multiply 函数### 练习2声明 `*.jpg` 模块 (默认导出 string)### 练习3声明全局变量 `API_BASE_URL: string` 和 `DEBUG_MODE: boolean`

In [ ]:
// 模拟声明文件内容 (TypeScript 中通过 declare module/global)// 练习1: 声明模块 (类型级别，编译时)/* // math-utils.d.tsdeclare module "math-utils" {  export function add(a: number, b: number): number;  export function subtract(a: number, b: number): number;  export function multiply(a: number, b: number): number;}*/// 练习2: 声明文件类型/*// images.d.tsdeclare module "*.jpg" {  const src: string;  export default src;}declare module "*.png" {  const src: string;  export default src;}*/// 练习3: 全局变量声明/*// globals.d.tsdeclare global {  const API_BASE_URL: string;  const DEBUG_MODE: boolean;}*/// 运行时验证声明概念type DeclaredAdd = (a: number, b: number) => number;const fakeAdd: DeclaredAdd = (a, b) => a + b;test("declare module concept", () => {  assert(fakeAdd(3, 4) === 7);});test("global type concept", () => {  const baseUrl: string = "https://api.example.com";  const debug: boolean = true;  assert(typeof baseUrl === "string" && debug === true);});runTests();

# Day 13 — 映射类型 (Mapped Types)---

## 📖 Mapped Types```ts// 基础映射: 把每个属性变为可选type MyPartial<T> = { [K in keyof T]?: T[K] };// 修饰符: -? 移除可选, -readonly 移除只读type MyRequired<T> = { [K in keyof T]-?: T[K] };type Mutable<T> = { -readonly [K in keyof T]: T[K] };// 带条件的映射: as 子句type Getters<T> = {  [K in keyof T as `get${Capitalize<string & K>}`]: () => T[K]};interface User {  name: string;  age: number;}type UserGetters = Getters<User>;// { getName: () => string; getAge: () => number }// 模板字面量类型 + 映射type EventName = "click" | "focus" | "blur";type EventHandler = {  [K in EventName as `on${Capitalize<K>}`]: (e: Event) => void};// { onClick: (e:Event)=>void; onFocus: ... }```

## ✏️ 练习### 练习1手写 `MyPick<T, K>` 映射类型### 练习2手写 `MyRecord<K, V>` 映射类型### 练习3`PrefixKeys<T, P>` 给所有属性加前缀: { prefix_key: T[key] }

In [ ]:
// 练习1: 手写 Pick (运行时通过普通函数模拟)function customPick<T extends object, K extends keyof T>(obj: T, keys: K[]): Pick<T, K> {  const result = {} as Pick<T, K>;  keys.forEach(key => { result[key] = obj[key]; });  return result;}test("customPick", () => {  const user = { id: 1, name: "小明", email: "test@test.com" };  const picked = customPick(user, ["name", "email"]);  assert("name" in picked && "email" in picked && !("id" in picked));});// 练习2: 手写 Recordfunction customRecord<K extends string, V>(keys: K[], valueFn: (key: K) => V): Record<K, V> {  const result = {} as Record<K, V>;  keys.forEach(k => { result[k] = valueFn(k); });  return result;}test("customRecord", () => {  const colors = customRecord(["red", "green", "blue"] as const, k => `#${k}`);  assert(colors.red === "#red" && colors.blue === "#blue");});// 练习3: PrefixKeys/*type PrefixKeys<T, P extends string> = {  [K in keyof T as `${P}${string & K}`]: T[K]};type Prefixed = PrefixKeys<{name:string;age:number}, "user_">;// { user_name: string; user_age: number }*/// 运行时验证function prefixKeys(obj: Record<string, any>, prefix: string): Record<string, any> {  const result: Record<string, any> = {};  for (const [k, v] of Object.entries(obj)) {    result[`${prefix}${k}`] = v;  }  return result;}test("prefixKeys", () => {  const prefixed = prefixKeys({ name: "x", age: 1 }, "user_");  assert(prefixed.user_name === "x" && prefixed.user_age === 1);});runTests();

# Day 14 — 条件类型 (Conditional Types)---

## 📖 Conditional Types```ts// 基本: T extends U ? X : Ytype IsNumber<T> = T extends number ? true : false;type A = IsNumber<42>;   // truetype B = IsNumber<"hi">; // false// 分布式条件类型 (union 会分别计算)type ToArray<T> = T extends any ? T[] : never;type C = ToArray<string | number>;  // string[] | number[]// infer 推断type ReturnOf<T> = T extends (...args: any[]) => infer R ? R : never;type D = ReturnOf<() => string>;  // stringtype FirstArg<T> = T extends (first: infer F, ...rest: any[]) => any ? F : never;// 实用: NonNullabletype NonNullable<T> = T extends null | undefined ? never : T;// 深层条件: 递归类型type DeepReadonly<T> = {  readonly [K in keyof T]: T[K] extends object ? DeepReadonly<T[K]> : T[K];};```

## ✏️ 练习### 练习1条件类型 `IsPromise<T>`: T extends Promise<any> → true : false### 练习2`Flatten<T>`: T extends (infer U)[] → U : T### 练习3`ExtractStringKeys<T>`: 提取所有值为 string 类型的属性名

In [ ]:
// 条件类型是编译时概念，运行时验证类型逻辑// 练习1: IsPromise — 运行时模拟function isPromise<T>(value: T): value is T extends Promise<any> ? T : never {  return value instanceof Promise;}test("isPromise", () => {  assert(isPromise(Promise.resolve(42)) === true);  assert(isPromise(42 as any) === false);});// 练习2: Flatten — 类型级/*type Flatten<T> = T extends (infer U)[] ? U : T;type F1 = Flatten<string[]>;  // stringtype F2 = Flatten<number>;    // number*/// 练习3: ExtractStringKeys/*type ExtractStringKeys<T> = {  [K in keyof T]: T[K] extends string ? K : never}[keyof T];interface Obj { a: string; b: number; c: string; }type StrKeys = ExtractStringKeys<Obj>; // "a" | "c"*/// 运行时模拟function extractStringKeys(obj: Record<string, any>): string[] {  return Object.entries(obj)    .filter(([_, v]) => typeof v === "string")    .map(([k]) => k);}test("extractStringKeys", () => {  const keys = extractStringKeys({ a: "hello", b: 42, c: "world" });  assert(keys.length === 2 && keys.includes("a") && keys.includes("c"));});runTests();

# Day 15 — tsconfig 与编译选项---

## 📖 tsconfig.json```json{  "compilerOptions": {    "target": "ES2022",    "module": "ESNext",    "moduleResolution": "bundler",    "strict": true,    "esModuleInterop": true,    "skipLibCheck": true,    "outDir": "./dist",    "rootDir": "./src",    "declaration": true,    "sourceMap": true,    "noUnusedLocals": true,    "noUnusedParameters": true,    "noImplicitReturns": true,    "paths": {      "@/*": ["./src/*"]    }  },  "include": ["src/**/*"],  "exclude": ["node_modules", "dist"]}```**关键选项**:- `strict: true` — 开启所有严格检查(推荐)- `target` — 编译到哪个 ES 版本- `module` — 模块系统 (ESNext/CommonJS)- `paths` — 路径别名映射

## ✏️ 练习### 练习1写一个 tsconfig.json 模板 (target ES2022, strict true, src→dist)### 练习2`strict` 包括哪些子选项？列出 3 个以上### 练习3解释 `declaration: true` 和 `sourceMap: true` 的作用

In [ ]:
// 练习1: tsconfigconst tsconfig = {  compilerOptions: {    target: "ES2022",    module: "ESNext",    strict: true,    esModuleInterop: true,    outDir: "./dist",    rootDir: "./src"  },  include: ["src/**/*"]};test("tsconfig structure", () => {  assert(tsconfig.compilerOptions.strict === true);  assert(tsconfig.compilerOptions.outDir === "./dist");});// 练习2: strict 子选项const strictSubOptions = [  "strictNullChecks",      // null/undefined 检查  "noImplicitAny",         // 禁止隐式 any  "strictFunctionTypes",   // 严格函数类型检查  "strictBindCallApply",   // 严格 bind/call/apply  "noImplicitThis"         // 禁止隐式 this];test("strict sub-options", () => {  assert(strictSubOptions.length >= 5);  assert(strictSubOptions.includes("noImplicitAny"));});// 练习3: declaration & sourceMap/*declaration: true → 生成 .d.ts 类型声明文件  - 允许其他 TS 项目使用你的库时有类型提示  - 输出在 outDir 中sourceMap: true → 生成 .js.map 文件  - 调试时可看到原始 TS 源码而非编译后的 JS  - 在浏览器 DevTools 中可直接断点 TS 文件*/test("declaration & sourceMap", () => {  const explanation = "declaration 生成 .d.ts 类型文件, sourceMap 用于调试";  assert(explanation.includes(".d.ts") && explanation.includes("调试"));});runTests();

# Day 16 — 模块系统---

## 📖 ES 模块 vs CommonJS```ts// ES 模块 (推荐)// math.tsexport function add(a: number, b: number): number { return a + b; }export const PI = 3.14159;export default class Calculator { }// app.tsimport Calculator, { add, PI } from "./math";import * as math from "./math";// 动态导入const module = await import("./heavy-module");// CommonJS (Node.js 传统)// const fs = require("fs");  // 类型: anyimport fs from "fs";            // 需要 esModuleInteropimport * as fs from "fs";       // 推荐// 模块解析策略// "node" — Node.js 风格 (node_modules)// "bundler" — Vite/esbuild 风格 (推荐新项目)// "classic" — 旧版 (不推荐)```

## ✏️ 练习### 练习1写一个 utils.ts 模块: export capitalize, truncate 函数### 练习2写一个默认导出类 ConfigManager: get, set, has 方法### 练习3export type + export interface 从 types.ts 导出，在 app.ts 中导入使用

In [ ]:
// 练习1: 模块内容// utils.ts contents:const utilsModule = {  capitalize(str: string): string {    return str.charAt(0).toUpperCase() + str.slice(1);  },  truncate(str: string, len: number): string {    return str.length > len ? str.slice(0, len) + "..." : str;  }};test("utils module", () => {  assert(utilsModule.capitalize("hello") === "Hello");  assert(utilsModule.truncate("hello world", 5) === "hello...");});// 练习2: 默认导出类class ConfigManager {  private config = new Map<string, unknown>();  set(key: string, value: unknown) { this.config.set(key, value); }  get<T>(key: string, defaultValue?: T): T | undefined {    return (this.config.get(key) as T) ?? defaultValue;  }  has(key: string): boolean { return this.config.has(key); }}test("ConfigManager", () => {  const cm = new ConfigManager();  cm.set("theme", "dark");  cm.set("timeout", 5000);  assert(cm.get<string>("theme") === "dark");  assert(cm.get<number>("timeout") === 5000);  assert(cm.has("theme") === true);  assert(cm.get<string>("missing", "default") === "default");});// 练习3: 类型导出// types.ts contents:interface AppConfig { apiUrl: string; debug: boolean; }type Theme = "light" | "dark" | "system";const config: AppConfig = { apiUrl: "https://api.example.com", debug: true };const theme: Theme = "dark";test("types module", () => {  assert(config.apiUrl.includes("https") && theme === "dark");});runTests();

# Day 17 — 可辨识联合 (Discriminated Unions)---

## 📖 Discriminated Unions```ts// 每个成员有相同的字面量属性 (判别键)type Shape =  | { kind: "circle"; radius: number }  | { kind: "rectangle"; width: number; height: number }  | { kind: "triangle"; base: number; height: number };function area(shape: Shape): number {  switch (shape.kind) {    case "circle":      return Math.PI * shape.radius ** 2;    case "rectangle":      return shape.width * shape.height;    case "triangle":      return (shape.base * shape.height) / 2;  }}// 实际场景: API 响应type ApiResponse<T> =  | { status: "loading" }  | { status: "success"; data: T }  | { status: "error"; error: string };function handleResponse<T>(resp: ApiResponse<T>): string {  switch (resp.status) {    case "loading": return "加载中...";    case "success": return `数据: ${JSON.stringify(resp.data)}`;    case "error": return `错误: ${resp.error}`;  }}```

## ✏️ 练习### 练习1Result<T> discriminated union: kind="ok"+value | kind="err"+message### 练习2HTTP 请求状态: idle+null | loading | success+T | error+string### 练习3事件系统: click+{x,y} | keypress+{key} | submit+{formId}

In [ ]:
// 练习1type Result<T> =  | { kind: "ok"; value: T }  | { kind: "err"; message: string };function processResult<T>(r: Result<T>): string {  switch (r.kind) {    case "ok": return `成功: ${r.value}`;    case "err": return `失败: ${r.message}`;  }}test("Result ok", () => {  assert(processResult({ kind: "ok", value: 42 }) === "成功: 42");});test("Result err", () => {  assert(processResult({ kind: "err", message: "超时" }) === "失败: 超时");});// 练习2type HttpState<T> =  | { status: "idle" }  | { status: "loading" }  | { status: "success"; data: T }  | { status: "error"; message: string };function renderState<T>(state: HttpState<T>): string {  switch (state.status) {    case "idle": return "等待请求";    case "loading": return "加载中...";    case "success": return `已加载: ${JSON.stringify(state.data)}`;    case "error": return `出错: ${state.message}`;  }}test("HttpState", () => {  assert(renderState({ status: "idle" }) === "等待请求");  assert(renderState({ status: "success", data: [1,2,3] }) === "已加载: [1,2,3]");});// 练习3type AppEvent =  | { type: "click"; x: number; y: number }  | { type: "keypress"; key: string }  | { type: "submit"; formId: string };function handleEvent(e: AppEvent): string {  switch (e.type) {    case "click": return `点击 (${e.x}, ${e.y})`;    case "keypress": return `按键: ${e.key}`;    case "submit": return `提交表单: ${e.formId}`;  }}test("AppEvent click", () => {  assert(handleEvent({ type: "click", x: 100, y: 200 }).includes("100"));});test("AppEvent keypress", () => {  assert(handleEvent({ type: "keypress", key: "Enter" }) === "按键: Enter");});runTests();

# Day 18 — 泛型递归与模板字面量---

## 📖 Advanced Types```ts// 递归类型type JSONValue =  | string  | number  | boolean  | null  | JSONValue[]  | { [key: string]: JSONValue };// 递归映射: DeepPartialtype DeepPartial<T> = {  [K in keyof T]?: T[K] extends object ? DeepPartial<T[K]> : T[K];};// 模板字面量类型type Greeting = `Hello, ${string}!`;  // "Hello, World!", "Hello, 小明!"type EventName = `on${Capitalize<string>}`;// 字符串操作类型type Shout<S extends string> = Uppercase<S>;type T1 = Shout<"hello">;  // "HELLO"// 组合: 路由参数提取type RouteParams<T extends string> =  T extends `${string}:${infer Param}/${infer Rest}`    ? Param | RouteParams<Rest>    : T extends `${string}:${infer Param}`      ? Param      : never;```

## ✏️ 练习### 练习1实现路由路径类型: `/users/:id/posts/:postId` 提取参数 id | postId### 练习2DeepReadonly<T> 递归实现深度只读### 练习3用模板字面量类型生成 CSS margin/padding 属性联合类型

In [ ]:
// 练习1: 路由参数提取 — 运行时模拟function extractParams(path: string): string[] {  const regex = /:(\w+)/g;  const params: string[] = [];  let match;  while ((match = regex.exec(path)) !== null) {    params.push(match[1]);  }  return params;}test("extract route params", () => {  const params = extractParams("/users/:id/posts/:postId");  assert(params.length === 2);  assert(params.includes("id") && params.includes("postId"));});// 练习2: DeepReadonly — 运行时模拟type DeepReadonlyObj<T> = {  readonly [K in keyof T]: T[K] extends object ? DeepReadonlyObj<T[K]> : T[K];};// 概念验证function deepFreeze<T extends object>(obj: T): Readonly<T> {  Object.freeze(obj);  for (const key in obj) {    if (typeof obj[key] === "object" && obj[key] !== null) {      deepFreeze(obj[key] as object);    }  }  return obj;}test("deepReadonly", () => {  const obj = deepFreeze({ a: 1, b: { c: 2 } });  assert(obj.a === 1 && obj.b.c === 2);});// 练习3: CSS 属性联合/*type MarginProperty = `margin-${"top" | "right" | "bottom" | "left"}`;type PaddingProperty = `padding-${"top" | "right" | "bottom" | "left"}`;type SpacingProperty = MarginProperty | PaddingProperty;*/function setSpacing(prop: string): boolean {  const valid = /^(margin|padding)-(top|right|bottom|left)$/.test(prop);  return valid;}test("CSS spacing type", () => {  assert(setSpacing("margin-top") === true);  assert(setSpacing("padding-left") === true);  assert(setSpacing("margin-center") === false);});runTests();

# Day 19 — Generics + React (概述)---

## 📖 TypeScript + React```tsx// React 组件类型import { FC, useState, useRef } from "react";// Props 接口interface ButtonProps {  label: string;  onClick: () => void;  variant?: "primary" | "secondary";  disabled?: boolean;}// 函数组件const Button: FC<ButtonProps> = ({ label, onClick, variant = "primary" }) => (  <button className={`btn-${variant}`} onClick={onClick}>    {label}  </button>);// useState 类型推导const [count, setCount] = useState<number>(0);const [name, setName] = useState("");  // 自动推导 string// useRef 泛型const inputRef = useRef<HTMLInputElement>(null);// 泛型组件interface ListProps<T> {  items: T[];  renderItem: (item: T) => React.ReactNode;}function List<T>({ items, renderItem }: ListProps<T>) {  return <ul>{items.map(renderItem)}</ul>;}```

## ✏️ 练习### 练习1写 InputProps 接口: value, onChange, placeholder?, type?### 练习2泛型 `useLocalStorage<T>(key, initial)` hook 的类型签名### 练习3`Event` 类型泛型处理: onClick(e: MouseEvent), onKeyDown(e: KeyboardEvent)

In [ ]:
// 练习1interface InputProps {  value: string;  onChange: (value: string) => void;  placeholder?: string;  type?: "text" | "password" | "email";}const inputProps: InputProps = {  value: "hello",  onChange: (v) => { /* setState */ },  placeholder: "请输入",  type: "text"};test("InputProps", () => {  assert(inputProps.type === "text" && inputProps.placeholder === "请输入");});// 练习2: useLocalStorage 类型function useLocalStorage<T>(key: string, initial: T): [T, (value: T) => void] {  return [initial, (v) => { /* localStorage.setItem(key, JSON.stringify(v)) */ }];}const [theme, setTheme] = useLocalStorage<string>("theme", "light");test("useLocalStorage generic", () => {  assert(theme === "light");  assert(typeof setTheme === "function");});// 练习3: 事件类型interface EventHandlers {  onClick?: (event: { type: string; target: unknown }) => void;  onKeyDown?: (event: { type: string; key: string }) => void;}function createHandler(type: string): (e: { type: string }) => string {  return (e) => `${type}: ${e.type}`;}const clickHandler = createHandler("click");const keyHandler = createHandler("keydown");test("Event handlers", () => {  assert(clickHandler({ type: "click" }) === "click: click");  assert(keyHandler({ type: "keydown" }) === "keydown: keydown");});runTests();

# Day 20 — Node.js + Express 类型---

## 📖 TypeScript + Express```tsimport express, { Request, Response, NextFunction } from "express";const app = express();// 扩展 Request 类型interface AuthenticatedRequest extends Request {  user?: { id: number; name: string };}// 路由处理app.get("/users/:id", (  req: Request<{ id: string }>,  // 路径参数  res: Response) => {  const userId = req.params.id;  res.json({ id: userId });});// 中间件function authMiddleware(  req: AuthenticatedRequest,  res: Response,  next: NextFunction) {  // 验证逻辑  req.user = { id: 1, name: "小明" };  next();}// 错误处理function errorHandler(  err: Error,  req: Request,  res: Response,  next: NextFunction) {  console.error(err.stack);  res.status(500).json({ error: err.message });}```

## ✏️ 练习### 练习1写 GET /api/products 的 handler 类型签名### 练习2定义一个 LoggerMiddleware 类型: (req, res, next) => void### 练习3用泛型写 `createRouter<T>()` 创建带类型约束的路由

In [ ]:
// 练习1: Handler 类型interface TypedRequest<P = {}, B = {}, Q = {}> {  params: P;  body: B;  query: Q;}interface TypedResponse<T> {  json(data: T): void;  status(code: number): TypedResponse<T>;}type GetProductsHandler = (  req: TypedRequest<{}, {}, { page?: string; limit?: string }>,  res: TypedResponse<{ products: string[]; total: number }>) => void;// 练习2type LoggerMiddleware<T = TypedRequest> = (  req: T,  res: TypedResponse<any>,  next: () => void) => void;function createLogger(): LoggerMiddleware {  return (req, res, next) => {    console.log(`[LOG] ${new Date().toISOString()}`);    next();  };}test("LoggerMiddleware", () => {  const mw = createLogger();  assert(typeof mw === "function");});// 练习3: 泛型 Routerclass TypedRouter<T extends Record<string, any>> {  private routes = new Map<string, (data: T) => unknown>();  add(method: string, path: string, handler: (data: T) => unknown) {    this.routes.set(`${method} ${path}`, handler);  }  handle(method: string, path: string, data: T): unknown {    const key = `${method} ${path}`;    const handler = this.routes.get(key);    if (!handler) throw new Error(`No route: ${key}`);    return handler(data);  }}interface UserData { id: number; name: string; }const router = new TypedRouter<UserData>();router.add("GET", "/users", (data) => [{ id: data.id, name: data.name }]);test("TypedRouter", () => {  const result = router.handle("GET", "/users", { id: 1, name: "小明" });  assert(Array.isArray(result));});runTests();

# Day 21 — 实战: 构建类型安全的 API 客户端---

## 📖 Typed API Client```ts// 定义 API 端点类型interface Endpoints {  "GET /users": { response: User[] };  "GET /users/:id": { params: { id: string }; response: User };  "POST /users": { body: CreateUser; response: User };}type User = { id: number; name: string; email: string };type CreateUser = Omit<User, "id">;// 类型安全的 fetch 封装async function apiClient<  P extends keyof Endpoints,  R extends Endpoints[P]["response"]>(  path: P,  options?: {    params?: P extends `${string}:${string}` ? Endpoints[P] extends { params: infer Params } ? Params : never : never;    body?: Endpoints[P] extends { body: infer Body } ? Body : never;  }): Promise<R> {  const response = await fetch(path as string);  return response.json();}// 使用 — 全类型安全const users = await apiClient("GET /users");const user = await apiClient("GET /users/:id", { params: { id: "1" } });```

## ✏️ 练习### 练习1定义 API 端点类型: GET /products, POST /products, GET /products/:id### 练习2实现 `typedGet<T>(url, params?): Promise<T>` 基础封装### 练习3添加错误处理: 统一返回 `ApiResponse<T>` 类型

In [ ]:
// 练习1interface Product { id: number; name: string; price: number; }interface ApiEndpoints {  getProducts: { response: Product[] };  getProduct: { params: { id: number }; response: Product };  createProduct: { body: Omit<Product, "id">; response: Product };}// 练习2async function typedGet<T>(url: string, params?: Record<string, string>): Promise<T> {  const queryString = params ? "?" + new URLSearchParams(params).toString() : "";  const response = await fetch(url + queryString);  if (!response.ok) throw new Error(`HTTP ${response.status}`);  return response.json() as Promise<T>;}// 练习3type ApiResponse2<T> =  | { ok: true; data: T }  | { ok: false; error: string; status: number };async function safeGet<T>(url: string): Promise<ApiResponse2<T>> {  try {    const resp = await fetch(url);    if (!resp.ok) return { ok: false, error: `HTTP ${resp.status}`, status: resp.status };    const data = await resp.json();    return { ok: true, data };  } catch (e) {    return { ok: false, error: e instanceof Error ? e.message : "未知错误", status: 0 };  }}// 概念验证test("ApiResponse types", () => {  const success: ApiResponse2<Product> = { ok: true, data: { id: 1, name: "test", price: 99 } };  const failure: ApiResponse2<Product> = { ok: false, error: "网络错误", status: 500 };  assert(success.ok === true && failure.ok === false);  assert("data" in success && "error" in failure);});runTests();

# Day 22 — 泛型约束组合实战---

## 📖 组合泛型实战```ts// 约束 + 默认值 + 推导 组合interface Repository<T extends { id: number }> {  findAll(): Promise<T[]>;  findById(id: T["id"]): Promise<T | null>;  create(data: Omit<T, "id">): Promise<T>;  update(id: T["id"], data: Partial<T>): Promise<T>;  delete(id: T["id"]): Promise<void>;}// 实现class InMemoryRepository<T extends { id: number }> implements Repository<T> {  private items: T[] = [];  private nextId = 1;  async findAll(): Promise<T[]> { return [...this.items]; }  async findById(id: number): Promise<T | null> {    return this.items.find(i => i.id === id) ?? null;  }  async create(data: Omit<T, "id">): Promise<T> {    const item = { ...data, id: this.nextId++ } as T;    this.items.push(item);    return item;  }  async delete(id: number): Promise<void> {    this.items = this.items.filter(i => i.id !== id);  }}```

## ✏️ 练习### 练习1实现泛型 `EventEmitter<Events>` 类: on, emit, off 方法### 练习2泛型 `Validator<T>` 类: addRule(field, validator), validate(data) → errors### 练习3泛型 `Cache<K, V>` 类: get, set, has, delete, clear 带 TTL

In [ ]:
// 练习1: EventEmittertype Listener<T extends any[]> = (...args: T) => void;class EventEmitter<Events extends Record<string, any[]>> {  private listeners = new Map<string, Listener<any>[]>();  on<E extends keyof Events>(event: E, listener: Listener<Events[E]>) {    const arr = this.listeners.get(event as string) ?? [];    arr.push(listener);    this.listeners.set(event as string, arr);  }  emit<E extends keyof Events>(event: E, ...args: Events[E]) {    this.listeners.get(event as string)?.forEach(fn => fn(...args));  }  off<E extends keyof Events>(event: E, listener: Listener<Events[E]>) {    const arr = this.listeners.get(event as string) ?? [];    this.listeners.set(event as string, arr.filter(fn => fn !== listener));  }}interface AppEvents {  login: [userId: number, name: string];  logout: [];  error: [message: string, code: number];}test("EventEmitter", () => {  const ee = new EventEmitter<AppEvents>();  let lastUser = "";  ee.on("login", (id, name) => { lastUser = name; });  ee.emit("login", 1, "小明");  assert(lastUser === "小明");});// 练习2: Validatorclass Validator<T extends Record<string, any>> {  private rules: Array<{ field: keyof T; validate: (value: T[keyof T]) => string | null }> = [];  addRule<K extends keyof T>(field: K, validate: (value: T[K]) => string | null) {    this.rules.push({ field, validate: validate as any });  }  validate(data: T): Record<string, string> {    const errors: Record<string, string> = {};    for (const rule of this.rules) {      const error = rule.validate(data[rule.field]);      if (error) errors[rule.field as string] = error;    }    return errors;  }}test("Validator", () => {  const v = new Validator<{ name: string; age: number }>();  v.addRule("name", val => val ? null : "姓名不能为空");  v.addRule("age", val => val > 0 ? null : "年龄必须大于0");    let errors = v.validate({ name: "", age: -1 });  assert(Object.keys(errors).length === 2);    errors = v.validate({ name: "小明", age: 25 });  assert(Object.keys(errors).length === 0);});// 练习3: Cache with TTLclass Cache<K, V> {  private store = new Map<K, { value: V; expiry: number }>();  set(key: K, value: V, ttlMs: number = 60000) {    this.store.set(key, { value, expiry: Date.now() + ttlMs });  }  get(key: K): V | null {    const entry = this.store.get(key);    if (!entry) return null;    if (Date.now() > entry.expiry) { this.store.delete(key); return null; }    return entry.value;  }  has(key: K): boolean { return this.get(key) !== null; }  delete(key: K) { this.store.delete(key); }  clear() { this.store.clear(); }}test("Cache with TTL", () => {  const cache = new Cache<string, number>();  cache.set("count", 42, 1000);  assert(cache.get("count") === 42);  assert(cache.has("count") === true);  cache.delete("count");  assert(cache.get("count") === null);});runTests();

# Day 23 — Zod 运行时验证---

## 📖 Zod — 类型安全的运行时验证```tsimport { z } from "zod";// 定义 schema (同时得到 TS 类型)const UserSchema = z.object({  name: z.string().min(2),  age: z.number().min(0).max(150),  email: z.string().email().optional()});type User = z.infer<typeof UserSchema>;// 运行时验证const result = UserSchema.safeParse(data);if (result.success) {  const user: User = result.data;} else {  console.error(result.error.errors);}// 复杂 schemaconst OrderSchema = z.object({  id: z.string().uuid(),  items: z.array(z.object({    productId: z.string(),    quantity: z.number().int().positive()  })),  status: z.enum(["pending","shipped","delivered"])});```

## ✏️ 练习### 练习1用 Zod schema 定义 Product (name, price正数, category可选)### 练习2CreateUser schema: name(2-50字), email(合法邮箱), password(至少8位)### 练习3FormData schema: 含嵌套对象 address{street, city, zipCode} 和 tags 数组

In [ ]:
// 模拟 Zod (不实际引入库)class ZodString {  min(len: number) { return this; }  max(len: number) { return this; }  email() { return this; }}class ZodNumber {  min(val: number) { return this; }  max(val: number) { return this; }  positive() { return this; }  int() { return this; }}const z = {  string: () => new ZodString(),  number: () => new ZodNumber(),  object: (schema: any) => ({ parse: (data: any) => data }),  enum: (values: readonly string[]) => values[0],  array: (item: any) => ([]),  infer: {} as any};// 练习1const ProductSchema = {  name: z.string().min(1),  price: z.number().positive(),  category: z.string().optional};test("Product schema", () => assert("name" in ProductSchema));// 练习2const CreateUserSchema = {  name: z.string().min(2).max(50),  email: z.string().email(),  password: z.string().min(8)};test("CreateUser schema", () => assert("password" in CreateUserSchema));// 练习3const FormSchema = {  address: z.object({    street: z.string(),    city: z.string(),    zipCode: z.string()  }),  tags: z.array(z.string())};test("Form schema nested", () => assert("address" in FormSchema && "tags" in FormSchema));runTests();

# Day 24 — 高级类型体操 1---

## 📖 Type Challenges (Easy/Medium)```ts// 实现 Picktype MyPick<T, K extends keyof T> = {  [P in K]: T[P];};// 实现 Readonlytype MyReadonly<T> = {  readonly [P in keyof T]: T[P];};// 实现 Excludetype MyExclude<T, U> = T extends U ? never : T;// 实现 Extracttype MyExtract<T, U> = T extends U ? T : never;// 实现 ReturnTypetype MyReturnType<T> = T extends (...args: any[]) => infer R ? R : never;// 实现 Omit (用 Pick + Exclude)type MyOmit<T, K extends keyof T> = Pick<T, Exclude<keyof T, K>>;```

## ✏️ 练习### 练习1手写 `DeepPick<T, K>` 实现深度 Pick### 练习2手写 `Merge<F, S>` 合并两个对象类型 (S 覆盖 F)### 练习3手写 `TupleToUnion<T>` 把元组转为联合类型

In [ ]:
// 类型体操是编译时概念，运行时验证逻辑// 练习1: DeepPick 思路/*type DeepPick<T, Path extends string> =  Path extends `${infer K}.${infer Rest}`    ? K extends keyof T ? { [P in K]: DeepPick<T[K], Rest> } : never    : Path extends keyof T ? { [P in Path]: T[Path] } : never;*/// 练习2: Mergefunction merge<F extends object, S extends object>(first: F, second: S): F & S {  return { ...first, ...second };}test("Merge types", () => {  const merged = merge({ a: 1, b: "hello" }, { b: 42, c: true });  assert(merged.a === 1 && merged.b === 42 && merged.c === true);});// 练习3: TupleToUnion/*type TupleToUnion<T extends any[]> = T[number];type T1 = TupleToUnion<[1, 2, 3]>; // 1 | 2 | 3*/function tupleToUnion<T extends any[]>(tuple: T): T[number] {  return tuple[0]; // 类型层面: T[number]}test("TupleToUnion concept", () => {  const val = tupleToUnion([1, 2, 3] as const);  assert(typeof val === "number");});runTests();

# Day 25 — 高级类型体操 2---

## 📖 Type Challenges (Hard)```ts// 深度只读type DeepReadonly<T> = {  readonly [K in keyof T]: T[K] extends object    ? T[K] extends Function ? T[K] : DeepReadonly<T[K]>    : T[K];};// 类型安全的 EventEmittertype EventMap = {  click: [x: number, y: number];  keydown: [key: string];};// Currying 类型type Curry<F extends (...args: any[]) => any> =  F extends (first: infer A, ...rest: infer R) => infer Ret    ? (arg: A) => Curry<(...args: R) => Ret>    : ReturnType<F>;// 类型安全的 Path 推导type Paths<T> = T extends object ? {  [K in keyof T]: `${string & K}` | `${string & K}.${Paths<T[K]>}`}[keyof T] : never;```

## ✏️ 练习### 练习1实现 `DeepRequired<T>` 所有属性(包括嵌套)变为必选### 练习2实现 `NonNullableKeys<T>` 提取所有非空值属性名### 练习3实现类型安全的 `get(obj, path)` 函数签名

In [ ]:
// 练习1: DeepRequired — 运行时模拟function deepRequired<T extends Record<string, any>>(obj: T, defaults: Partial<T>): Required<T> {  const result = { ...defaults, ...obj } as Required<T>;  for (const key of Object.keys(result)) {    if (result[key] === undefined) throw new Error(`Missing: ${key}`);  }  return result;}test("deepRequired", () => {  const result = deepRequired({ name: "test" }, { name: "default", age: 0 });  assert(result.name === "test" && result.age === 0);});// 练习2: NonNullableKeysfunction nonNullableKeys<T extends Record<string, any>>(obj: T): Array<keyof T> {  return Object.entries(obj)    .filter(([_, v]) => v !== null && v !== undefined)    .map(([k]) => k as keyof T);}test("nonNullableKeys", () => {  const obj = { a: "hello", b: null, c: undefined, d: 42 };  const keys = nonNullableKeys(obj);  assert(keys.length === 2 && keys.includes("a") && keys.includes("d"));});// 练习3: typed getfunction typedGet<T extends Record<string, any>, K extends keyof T>(obj: T, key: K): T[K] {  return obj[key];}function typedGetPath<T extends Record<string, any>>(obj: T, path: string): any {  return path.split(".").reduce((acc: any, key) => acc?.[key], obj);}test("typedGet", () => {  const obj = { user: { name: "小明", age: 25 } };  assert(typedGet(obj, "user").name === "小明");  assert(typedGetPath(obj, "user.name") === "小明");});runTests();

# Day 26 — Monorepo 与构建工具---

## 📖 tsc / esbuild / tsup```bash# TypeScript 编译器npx tsc                    # 编译npx tsc --watch            # 监听模式npx tsc --noEmit           # 仅类型检查(不输出)# esbuild (快速打包)npx esbuild src/index.ts --bundle --outfile=dist/bundle.js# tsup (基于 esbuild 的 TS 打包器)npx tsup src/index.ts --format cjs,esm --dts# tsx (直接运行 TS)npx tsx src/server.ts```**常见项目结构**:```monorepo/├── packages/│   ├── shared/      # 共享类型和工具│   ├── server/      # 后端│   └── client/      # 前端├── tsconfig.base.json└── package.json```

## ✏️ 练习### 练习1写一个 package.json scripts: build(tsc), dev(tsx watch), typecheck(tsc --noEmit)### 练习2写 tsconfig 引用 (references): base → server/client### 练习3实现 `createPackage(name)` 工厂函数生成包配置

In [ ]:
// 练习1const scripts = {  build: "tsc",  dev: "tsx watch src/index.ts",  typecheck: "tsc --noEmit",  lint: "eslint src/",  test: "vitest"};test("package scripts", () => {  assert(scripts.build === "tsc");  assert(scripts.dev.includes("tsx"));  assert(scripts.typecheck.includes("noEmit"));});// 练习2: tsconfig referencesconst tsconfigBase = {  compilerOptions: { strict: true, target: "ES2022" }};const tsconfigServer = {  extends: "./tsconfig.base.json",  compilerOptions: { outDir: "./dist" },  references: [{ path: "../shared" }]};test("tsconfig references", () => {  assert("references" in tsconfigServer);  assert(tsconfigServer.extends.includes("base"));});// 练习3interface PackageConfig {  name: string;  version: string;  main: string;  types: string;}function createPackage(name: string): PackageConfig {  return {    name: `@acme/${name}`,    version: "1.0.0",    main: `./dist/${name}.js`,    types: `./dist/${name}.d.ts`  };}test("createPackage", () => {  const pkg = createPackage("utils");  assert(pkg.name === "@acme/utils" && pkg.types.endsWith(".d.ts"));});runTests();

# Day 27 — 异常处理与 Result 模式---

## 📖 Result / Either 模式```ts// Result 类型 — 替代 try/catchtype Result<T, E = Error> =  | { ok: true; value: T }  | { ok: false; error: E };// 工厂函数function success<T>(value: T): Result<T, never> {  return { ok: true, value };}function failure<E>(error: E): Result<never, E> {  return { ok: false, error };}// 组合器function map<T, U, E>(result: Result<T, E>, fn: (value: T) => U): Result<U, E> {  if (result.ok) return success(fn(result.value));  return result;}function flatMap<T, U, E>(result: Result<T, E>, fn: (value: T) => Result<U, E>): Result<U, E> {  if (result.ok) return fn(result.value);  return result;}// 安全包装async function toResult<T>(promise: Promise<T>): Promise<Result<T, Error>> {  try {    const value = await promise;    return success(value);  } catch (e) {    return failure(e instanceof Error ? e : new Error(String(e)));  }}```

## ✏️ 练习### 练习1实现 `unwrap(result, defaultValue)` 从 Result 取值### 练习2实现 `mapError(result, fn)` 转换错误类型### 练习3链式处理: parse → validate → save，每步返回 Result，用 flatMap 串联

In [ ]:
type Result<T, E = string> =  | { ok: true; value: T }  | { ok: false; error: E };function success<T>(value: T): Result<T, never> {  return { ok: true, value };}function failure<E>(error: E): Result<never, E> {  return { ok: false, error };}// 练习1function unwrap<T, E>(result: Result<T, E>, defaultValue: T): T {  return result.ok ? result.value : defaultValue;}test("unwrap", () => {  assert(unwrap(success(42), 0) === 42);  assert(unwrap(failure("error"), 0) === 0);});// 练习2function mapError<T, E, F>(result: Result<T, E>, fn: (error: E) => F): Result<T, F> {  if (result.ok) return result;  return failure(fn(result.error));}test("mapError", () => {  const r = mapError(failure("low-level"), (e) => `Wrapped: ${e}`);  assert(!r.ok && r.error === "Wrapped: low-level");});// 练习3: 链式处理function flatMap<T, U, E>(result: Result<T, E>, fn: (value: T) => Result<U, E>): Result<U, E> {  if (result.ok) return fn(result.value);  return result;}function parse(s: string): Result<number, string> {  const n = Number(s);  return isNaN(n) ? failure("解析失败") : success(n);}function validate(n: number): Result<number, string> {  return n > 0 ? success(n) : failure("必须大于0");}function save(n: number): Result<string, string> {  return success(`已保存: ${n}`);}test("Result chain", () => {  const r = flatMap(flatMap(parse("42"), validate), save);  assert(r.ok && r.value === "已保存: 42");});test("Result chain failure", () => {  const r = flatMap(flatMap(parse("abc"), validate), save);  assert(!r.ok && r.error === "解析失败");});runTests();

# Day 28 — 设计模式 (TS 版)---

## 📖 常见设计模式```ts// 1. 单例 (Singleton)class Database {  private static instance: Database;  private constructor() {}  static getInstance(): Database {    if (!Database.instance) Database.instance = new Database();    return Database.instance;  }}// 2. 工厂 (Factory)interface Button { render(): string; }class PrimaryButton implements Button { render() { return "<primary>"; } }class SecondaryButton implements Button { render() { return "<secondary>"; } }class ButtonFactory {  static create(type: "primary" | "secondary"): Button {    return type === "primary" ? new PrimaryButton() : new SecondaryButton();  }}// 3. 策略 (Strategy)type SortStrategy = (arr: number[]) => number[];const quickSort: SortStrategy = (arr) => [...arr].sort((a,b) => a-b);const reverseSort: SortStrategy = (arr) => [...arr].sort((a,b) => b-a);class Sorter {  constructor(private strategy: SortStrategy) {}  sort(arr: number[]): number[] { return this.strategy(arr); }}```

## ✏️ 练习### 练习1实现 `Observer` 模式: Subject 类 (subscribe/unsubscribe/notify)### 练习2实现 `Builder` 模式: QueryBuilder 链式构建 SQL### 练习3实现 `Decorator` 模式: 给 Repository 加缓存/日志装饰器

In [ ]:
// 练习1: Observertype Observer<T> = (data: T) => void;class Subject<T> {  private observers: Observer<T>[] = [];  subscribe(observer: Observer<T>) { this.observers.push(observer); }  unsubscribe(observer: Observer<T>) {    this.observers = this.observers.filter(o => o !== observer);  }  notify(data: T) { this.observers.forEach(o => o(data)); }}test("Observer", () => {  const subject = new Subject<string>();  let received = "";  const observer = (msg: string) => { received = msg; };  subject.subscribe(observer);  subject.notify("hello");  assert(received === "hello");  subject.unsubscribe(observer);  subject.notify("world");  assert(received === "hello"); // unchanged});// 练习2: Builderclass QueryBuilder {  private tableName = "";  private columns = ["*"];  private conditions: string[] = [];  private orderField = "";  private limitValue = 0;  from(table: string): this { this.tableName = table; return this; }  select(...cols: string[]): this { this.columns = cols; return this; }  where(condition: string): this { this.conditions.push(condition); return this; }  orderBy(field: string): this { this.orderField = field; return this; }  limit(n: number): this { this.limitValue = n; return this; }  build(): string {    let sql = `SELECT ${this.columns.join(", ")} FROM ${this.tableName}`;    if (this.conditions.length) sql += ` WHERE ${this.conditions.join(" AND ")}`;    if (this.orderField) sql += ` ORDER BY ${this.orderField}`;    if (this.limitValue) sql += ` LIMIT ${this.limitValue}`;    return sql;  }}test("QueryBuilder", () => {  const sql = new QueryBuilder()    .from("users")    .select("id", "name")    .where("age > 18")    .orderBy("name")    .limit(10)    .build();  assert(sql === "SELECT id, name FROM users WHERE age > 18 ORDER BY name LIMIT 10");});// 练习3: Decoratorinterface Repo<T> {  get(id: number): T | null;  getAll(): T[];  save(item: T): void;}class InMemoryRepo<T extends { id: number }> implements Repo<T> {  private items: T[] = [];  get(id: number) { return this.items.find(i => i.id === id) ?? null; }  getAll() { return [...this.items]; }  save(item: T) {     const idx = this.items.findIndex(i => i.id === item.id);    if (idx >= 0) this.items[idx] = item;    else this.items.push(item);  }}class CachedRepo<T extends { id: number }> implements Repo<T> {  private cache = new Map<number, T>();  constructor(private inner: Repo<T>) {}  get(id: number): T | null {    if (this.cache.has(id)) { console.log("cache hit"); return this.cache.get(id)!; }    const item = this.inner.get(id);    if (item) this.cache.set(id, item);    return item;  }  getAll() { return this.inner.getAll(); }  save(item: T) { this.inner.save(item); this.cache.set(item.id, item); }}test("CachedRepo", () => {  const repo = new CachedRepo(new InMemoryRepo<{ id: number; name: string }>());  repo.save({ id: 1, name: "test" });  const item = repo.get(1);  assert(item?.name === "test");});runTests();

# Day 29 — 测试 (Vitest + Testing Library)---

## 📖 Vitest 测试框架```tsimport { describe, it, expect, beforeEach, vi } from "vitest";describe("Calculator", () => {  let calculator: Calculator;  beforeEach(() => {    calculator = new Calculator();  });  it("adds two numbers", () => {    expect(calculator.add(1, 2)).toBe(3);  });  it("handles async", async () => {    const result = await calculator.asyncAdd(1, 2);    expect(result).toBe(3);  });  // Mock  it("mocks fetch", async () => {    const mockFetch = vi.fn().mockResolvedValue({      json: () => Promise.resolve({ data: "test" })    });    const result = await fetchData(mockFetch);    expect(result.data).toBe("test");    expect(mockFetch).toHaveBeenCalledTimes(1);  });});```

## ✏️ 练习### 练习1为 `divide(a,b)` 写测试: 正常、除零、类型错误### 练习2用 mock 测试 `fetchUser(id)` 函数### 练习3集成测试: UserService (创建→查询→更新→删除)

In [ ]:
// 迷你测试框架 (不需 Vitest)class TestRunner {  private tests: Array<{ name: string; fn: () => void | Promise<void> }> = [];  private befores: Array<() => void> = [];  describe(name: string, fn: () => void) { fn(); }  it(name: string, fn: () => void | Promise<void>) { this.tests.push({ name, fn }); }  beforeEach(fn: () => void) { this.befores.push(fn); }  async run() {    let passed = 0, failed = 0;    for (const t of this.tests) {      try {        this.befores.forEach(fn => fn());        await t.fn();        passed++;        console.log(`  ✅ ${t.name}`);      } catch (e) {        failed++;        console.log(`  ❌ ${t.name}: ${e}`);      }    }    console.log(`\n📊 ${passed}/${passed + failed} passed`);  }}const runner = new TestRunner();// 练习1function divide2(a: number, b: number): number {  if (b === 0) throw new Error("除零");  return a / b;}runner.it("divide normal", () => { assert(divide2(10, 2) === 5); });runner.it("divide by zero", () => {  try { divide2(1, 0); assert(false, "should throw"); } catch { /* ok */ }});// 练习2async function fetchUser(id: number): Promise<{ id: number; name: string }> {  const resp = await fetch(`/users/${id}`);  return resp.json();}runner.it("fetchUser mock", async () => {  let calledUrl = "";  const mockFetch = (url: string) => {    calledUrl = url;    return Promise.resolve({ json: () => Promise.resolve({ id: 1, name: "test" }) });  };  // 模拟  const result = await mockFetch("/users/1").then((r: any) => r.json());  assert(result.id === 1 && result.name === "test");});// 练习3: UserServiceclass UserService {  private users: Map<number, { id: number; name: string }> = new Map();  create(name: string) {    const id = this.users.size + 1;    const user = { id, name };    this.users.set(id, user);    return user;  }  get(id: number) { return this.users.get(id); }  update(id: number, name: string) {    const user = this.users.get(id);    if (!user) throw new Error("not found");    user.name = name;    return user;  }  delete(id: number) { return this.users.delete(id); }}runner.it("UserService CRUD", () => {  const svc = new UserService();  const u = svc.create("小明");  assert(u.id === 1 && u.name === "小明");    const found = svc.get(1);  assert(found?.name === "小明");    svc.update(1, "大明");  assert(svc.get(1)?.name === "大明");    svc.delete(1);  assert(svc.get(1) === undefined);});runner.run();

# Day 30 — 综合项目: 类型安全的 Todo CLI---

## 📖 综合实战: TypeScript CLI 工具```tsimport { Command } from "commander";interface Todo {  id: number;  title: string;  done: boolean;  createdAt: Date;  priority: "low" | "medium" | "high";}class TodoService {  private todos: Todo[] = [];  private idCounter = 1;  add(title: string, priority: Todo["priority"] = "medium"): Todo {    const todo: Todo = {      id: this.idCounter++,      title,      done: false,      createdAt: new Date(),      priority    };    this.todos.push(todo);    return todo;  }  list(filter?: Todo["priority"]): Todo[] {    return filter ? this.todos.filter(t => t.priority === filter) : [...this.todos];  }  done(id: number): Todo | null {    const todo = this.todos.find(t => t.id === id);    if (todo) todo.done = true;    return todo ?? null;  }  stats() {    const total = this.todos.length;    const done = this.todos.filter(t => t.done).length;    return { total, done, pending: total - done };  }}```**30天回顾**: 基础类型→接口→泛型→联合类型→条件类型→映射类型→工具类型→类型守卫→模块→设计模式→项目实战

## ✏️ 练习### 练习1给 Todo 添加 `delete(id)` 和 `update(id, patch: Partial<Todo>)` 方法### 练习2添加 `search(query: string)` 搜索标题(不区分大小写)### 练习3添加 `sort(by: "date" | "priority" | "done")` 排序功能### 练习4数据持久化: `saveToFile(path)` 和 `loadFromFile(path)` (JSON)

In [ ]:
interface Todo {  id: number;  title: string;  done: boolean;  createdAt: Date;  priority: "low" | "medium" | "high";}class TodoService {  todos: Todo[] = [];  private idCounter = 1;  add(title: string, priority: Todo["priority"] = "medium"): Todo {    const todo: Todo = { id: this.idCounter++, title, done: false, createdAt: new Date(), priority };    this.todos.push(todo);    return todo;  }  delete(id: number): boolean {    const idx = this.todos.findIndex(t => t.id === id);    if (idx === -1) return false;    this.todos.splice(idx, 1);    return true;  }  update(id: number, patch: Partial<Pick<Todo, "title" | "priority" | "done">>): Todo | null {    const todo = this.todos.find(t => t.id === id);    if (!todo) return null;    Object.assign(todo, patch);    return todo;  }  search(query: string): Todo[] {    const q = query.toLowerCase();    return this.todos.filter(t => t.title.toLowerCase().includes(q));  }  sort(by: "date" | "priority" | "done"): Todo[] {    const order = { low: 0, medium: 1, high: 2 };    return [...this.todos].sort((a, b) => {      if (by === "date") return a.createdAt.getTime() - b.createdAt.getTime();      if (by === "done") return Number(a.done) - Number(b.done);      return order[a.priority] - order[b.priority];    });  }  stats() {    const total = this.todos.length;    const done = this.todos.filter(t => t.done).length;    return { total, done, pending: total - done };  }}// === TESTS ===const svc = new TodoService();svc.add("学TypeScript", "high");svc.add("运动", "medium");svc.add("读书", "low");svc.add("写代码", "high");svc.update(2, { done: true });test("delete", () => {  svc.delete(3);  assert(svc.todos.length === 3);});test("search", () => {  const results = svc.search("type");  assert(results.length === 1 && results[0].title === "学TypeScript");});test("sort by priority", () => {  const sorted = svc.sort("priority");  assert(sorted[0].priority === "low" || sorted[0].priority === "medium");  assert(sorted[sorted.length - 1].priority === "high");});test("sort by done", () => {  const sorted = svc.sort("done");  assert(sorted[sorted.length - 1].done === true); // 运动 done=true 在最后});test("stats", () => {  const s = svc.stats();  assert(s.total === 3);  assert(s.done >= 1);});test("update", () => {  const updated = svc.update(4, { title: "重构代码" });  assert(updated?.title === "重构代码");});console.log("🎉 Day 30 通过！");console.log("🏆 恭喜完成 30 天 TypeScript 核心训练！");console.log("📚 掌握: 类型系统→接口→泛型→联合/交叉→条件→映射→工具类型→设计模式→项目");console.log("🚀 下一步: 深入类型体操, 看 DefinitelyTyped 源码, 贡献开源!");runTests();

# 🏁 30 天 TypeScript 训练完成！**类型系统**: 基础类型→接口→泛型→联合→条件→映射→工具类型→类型体操→设计模式→项目**TypeScript = JavaScript + 类型安全 + 更好的开发体验 🦞**